In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialTemporalCore(nn.Module):
    def __init__(self, in_channels=1, num_freq_bins=128):
        super(SpatialTemporalCore, self).__init__()

        # ----------------------------------------------------
        # 1. SPATIAL CNN BLOCKS
        # Input shape: (Batch, in_channels, Freq, Time)
        # ----------------------------------------------------
        self.spatial_cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)), # Reduces Freq and Time by 2

            # Block 2
            nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)), # Reduces Freq and Time by 2

            # Block 3
            nn.Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Pool only along the frequency dimension this time to preserve temporal resolution
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        )

        # Calculate the remaining frequency bins after the MaxPool2d operations
        # 128 -> 64 (Block 1) -> 32 (Block 2) -> 16 (Block 3)
        reduced_freq_bins = num_freq_bins // 8

        # The feature dimension after flattening Channels and Freq
        cnn_output_dim = 128 * reduced_freq_bins

        # ----------------------------------------------------
        # 2. TEMPORAL DENSE BLOCKS
        # Refines the flattened spatial features at each time step
        # ----------------------------------------------------
        self.temporal_dense = nn.Sequential(
            nn.Linear(cnn_output_dim, 256),
            nn.BatchNorm1d(256), # Note: Will need shape adjustment during forward pass
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU()
        )

    def forward(self, x):
        """
        x shape expected from Member 1's Dataloader: (Batch, Channels, Freq, Time)
        """
        # --- 1. Pass through Spatial CNN ---
        # Shape goes from (B, C, F, T) -> (B, 128, F', T')
        x = self.spatial_cnn(x)

        # --- 2. Shape-Matching ---
        # We need to blend Channels and Freq, and swap Time to be the sequence axis.
        # Current shape: (B, Channels, Freq, Time)
        B, C, F, T = x.size()

        # Permute to: (Batch, Time, Channels, Freq)
        x = x.permute(0, 3, 1, 2).contiguous()

        # Flatten Channels and Freq into a single feature dimension
        # New shape: (Batch, Time, Channels * Freq)
        x = x.view(B, T, C * F)

        # --- 3. Pass through Temporal Dense Block ---
        # To apply BatchNorm1d correctly to sequence data, we temporarily reshape
        x = x.view(-1, C * F)             # Shape: (Batch * Time, Features)
        x = self.temporal_dense[0](x)     # Linear 1
        x = self.temporal_dense[1](x)     # BatchNorm1d
        x = self.temporal_dense[2](x)     # ReLU
        x = self.temporal_dense[3](x)     # Dropout
        x = self.temporal_dense[4](x)     # Linear 2
        x = self.temporal_dense[5](x)     # ReLU

        # Reshape back to sequence format
        x = x.view(B, T, -1)              # Shape: (Batch, Time, 128)

        # Output shape is now perfect for Member 3's Bi-LSTM + Attention block!
        return x